# Download ERA5 Reanalysis — ORCESTRA Domain

Downloads ERA5 data from the Copernicus Climate Data Store (CDS) for the ORCESTRA campaign domain and period.

| Download | Variables | Purpose |
|---|---|---|
| **A — Pressure levels** | `w` (omega), `t` (temperature) | Stitch above BEACH profiles in `boundary_fixes.ipynb` |
| **B — Single levels** | `sst`, `tcwv`, `u`/`v` at 200 & 850 hPa | RQ2 environmental context |

**Domain:** 0°N – 30°N, 70°W – 0°W  
**Period:** 2024-08-10 to 2024-09-30  
**Output:** `/g/data/k10/zr7147/ERA5/`

---

### Prerequisites — CDS API key

1. Register at [https://cds.climate.copernicus.eu](https://cds.climate.copernicus.eu)
2. Go to your profile → **API key** section
3. Create `~/.cdsapirc` with the content shown in **Section 0** below

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pathlib
import cdsapi

from scripts.config import default_era5_dir, default_era5_omega_path

In [ ]:

cdsapirc = pathlib.Path.home() / '.cdsapirc'
if cdsapirc.exists():
    print(f'Credentials file found: {cdsapirc}')
else:
    print('=' * 60)
    print('~/.cdsapirc NOT FOUND — create it before running downloads.')

In [ ]:
# ── Output directory ─────────────────────────────────────────────────────────
ERA5_DIR = default_era5_dir()
ERA5_DIR.mkdir(parents=True, exist_ok=True)
print(f'ERA5 output directory: {ERA5_DIR}')

In [ ]:
# ── Campaign parameters (shared by all downloads) ────────────────────────────
CAMPAIGN_START = '2024-08-10'
CAMPAIGN_END   = '2024-09-30'

# CDS area format: [North, West, South, East]
AREA = [30, -70, 0, 0]

# All dates in the campaign period
import pandas as pd
dates = pd.date_range(CAMPAIGN_START, CAMPAIGN_END, freq='D')
DATE_LIST = dates.strftime('%Y-%m-%d').tolist()
print(f'Dates: {DATE_LIST[0]} to {DATE_LIST[-1]}  ({len(DATE_LIST)} days)')
print(f'Domain: N{AREA[0]} W{abs(AREA[1])} S{AREA[2]} E{AREA[3]}')

## 1. Download A — Pressure-level omega and temperature

Used by `scripts/era5_extension.py` to stitch above BEACH profiles.

Output: `era5_omega_pressure_levels.nc`

**Why month-by-month?** A single request spanning the full campaign (52 days × 31 levels × 2 variables × 4 times/day) exceeds the CDS field limit. Splitting by month keeps each request well within the limit, then the monthly files are merged with xarray.

In [ ]:
PRESSURE_LEVELS = [
    '20', '30', '50', '70', '100', '125', '150', '175',
    '200', '225', '250', '300', '350', '400', '450',
    '500', '550', '600', '650', '700', '750', '775',
    '800', '825', '850', '875', '900', '925', '950', '975', '1000',
]

TIMES_PLEV = ['00:00', '06:00', '12:00', '18:00']

OUT_PLEV     = ERA5_DIR / 'era5_omega_pressure_levels.nc'
OUT_PLEV_TMP = ERA5_DIR / 'monthly_plev'   # staging folder for monthly chunks
OUT_PLEV_TMP.mkdir(parents=True, exist_ok=True)

print(f'Final output  : {OUT_PLEV}')
print(f'Monthly chunks: {OUT_PLEV_TMP}')
print(f'Levels: {len(PRESSURE_LEVELS)}  |  Times per day: {len(TIMES_PLEV)}')

In [ ]:
if OUT_PLEV.exists():
    print(f'Merged file already exists: {OUT_PLEV} ({OUT_PLEV.stat().st_size / 1e9:.2f} GB) — skipping.')
else:
    c = cdsapi.Client()
    months = sorted({(d[:4], d[5:7]) for d in DATE_LIST})
    monthly_files = []

    for year, month in months:
        out_m = OUT_PLEV_TMP / f'era5_plev_{year}_{month}.nc'
        monthly_files.append(out_m)

        if out_m.exists():
            print(f'  {year}-{month}: already downloaded ({out_m.stat().st_size / 1e6:.0f} MB)')
            continue

        days = sorted({d[8:10] for d in DATE_LIST if d[:4] == year and d[5:7] == month})
        print(f'  {year}-{month}: requesting {len(days)} days × {len(PRESSURE_LEVELS)} levels × 2 vars ...')

        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'variable': ['vertical_velocity', 'temperature'],
                'pressure_level': PRESSURE_LEVELS,
                'year':  year,
                'month': month,
                'day':   days,
                'time':  TIMES_PLEV,
                'area':  AREA,
                'format': 'netcdf',
            },
            str(out_m),
        )
        print(f'    Saved → {out_m}  ({out_m.stat().st_size / 1e6:.0f} MB)')

    print('\nMerging monthly files ...')
    import xarray as xr
    ds_merged = xr.open_mfdataset([str(f) for f in monthly_files], combine='by_coords')
    ds_merged.to_netcdf(str(OUT_PLEV))
    ds_merged.close()
    print(f'Merged → {OUT_PLEV}  ({OUT_PLEV.stat().st_size / 1e9:.2f} GB)')

In [ ]:
if OUT_PLEV.exists():
    import xarray as xr
    ds_plev = xr.open_dataset(OUT_PLEV)
    level_coord = 'pressure_level' if 'pressure_level' in ds_plev.coords else 'level'
    print(f'Variables : {list(ds_plev.data_vars)}')
    print(f'Time range: {str(ds_plev.time.values[0])[:16]} to {str(ds_plev.time.values[-1])[:16]}')
    print(f'Levels    : {ds_plev[level_coord].values} hPa')
    print(f'Lat range : {float(ds_plev.latitude.min()):.1f} to {float(ds_plev.latitude.max()):.1f}')
    print(f'Lon range : {float(ds_plev.longitude.min()):.1f} to {float(ds_plev.longitude.max()):.1f}')
    print(f'File size : {OUT_PLEV.stat().st_size / 1e9:.2f} GB')
    ds_plev.close()
else:
    print('File not yet downloaded.')

## 2. Download B — Single-level variables for RQ2 environmental context

Used to characterise the large-scale environment for each circle category.

| Variable | CDS name | Meaning |
|---|---|---|
| `sst` | `sea_surface_temperature` | SST |
| `tcwv` | `total_column_water_vapour` | Column water vapour |
| `u200`, `v200` | `u_component_of_wind`, `v_component_of_wind` at 200 hPa | Upper-level divergence |
| `u850`, `v850` | same at 850 hPa | Low-level convergence |

Output: `era5_single_levels.nc` and `era5_wind_200_850.nc`

In [ ]:
TIMES_SLEV = ['00:00', '06:00', '12:00', '18:00']

OUT_SLEV      = ERA5_DIR / 'era5_single_levels.nc'
OUT_WIND      = ERA5_DIR / 'era5_wind_200_850.nc'
OUT_SLEV_TMP  = ERA5_DIR / 'monthly_slev'
OUT_WIND_TMP  = ERA5_DIR / 'monthly_wind'
OUT_SLEV_TMP.mkdir(parents=True, exist_ok=True)
OUT_WIND_TMP.mkdir(parents=True, exist_ok=True)

print(f'Single-level output : {OUT_SLEV}')
print(f'Wind output         : {OUT_WIND}')

In [ ]:
import xarray as xr

def _download_monthly(c, dataset, request_base, date_list, out_tmp, out_final, label):
    """Download month-by-month into out_tmp, then merge into out_final."""
    if out_final.exists():
        print(f'{label}: already exists ({out_final.stat().st_size / 1e6:.0f} MB) — skipping.')
        return

    months = sorted({(d[:4], d[5:7]) for d in date_list})
    monthly_files = []

    for year, month in months:
        out_m = out_tmp / f'{out_tmp.name}_{year}_{month}.nc'
        monthly_files.append(out_m)

        if out_m.exists():
            print(f'  {label} {year}-{month}: already downloaded ({out_m.stat().st_size / 1e6:.0f} MB)')
            continue

        days = sorted({d[8:10] for d in date_list if d[:4] == year and d[5:7] == month})
        print(f'  {label} {year}-{month}: {len(days)} days ...')
        req = {**request_base, 'year': year, 'month': month, 'day': days}
        c.retrieve(dataset, req, str(out_m))
        print(f'    Saved ({out_m.stat().st_size / 1e6:.0f} MB)')

    print(f'  Merging {len(monthly_files)} monthly files ...')
    ds = xr.open_mfdataset([str(f) for f in monthly_files], combine='by_coords')
    ds.to_netcdf(str(out_final))
    ds.close()
    print(f'  Done → {out_final}  ({out_final.stat().st_size / 1e6:.0f} MB)')

In [ ]:
c = cdsapi.Client()

_download_monthly(
    c,
    dataset='reanalysis-era5-single-levels',
    request_base={
        'product_type': 'reanalysis',
        'variable': ['sea_surface_temperature', 'total_column_water_vapour'],
        'time': TIMES_SLEV,
        'area': AREA,
        'format': 'netcdf',
    },
    date_list=DATE_LIST,
    out_tmp=OUT_SLEV_TMP,
    out_final=OUT_SLEV,
    label='SST+TCWV',
)

In [ ]:
_download_monthly(
    c,
    dataset='reanalysis-era5-pressure-levels',
    request_base={
        'product_type': 'reanalysis',
        'variable': ['u_component_of_wind', 'v_component_of_wind'],
        'pressure_level': ['200', '850'],
        'time': TIMES_SLEV,
        'area': AREA,
        'format': 'netcdf',
    },
    date_list=DATE_LIST,
    out_tmp=OUT_WIND_TMP,
    out_final=OUT_WIND,
    label='Wind 200/850',
)

In [ ]:
# Verify single-level files
import xarray as xr

for label, path in [('Single-level (SST + TCWV)', OUT_SLEV),
                     ('Wind 200/850 hPa',          OUT_WIND)]:
    if path.exists():
        ds = xr.open_dataset(path)
        print(f'{label}:')
        print(f'  Variables  : {list(ds.data_vars)}')
        print(f'  Time range : {str(ds.time.values[0])[:16]} to {str(ds.time.values[-1])[:16]}')
        print(f'  Shape (t,lat,lon): {ds.time.size} × {ds.latitude.size} × {ds.longitude.size}')
        ds.close()
        print()
    else:
        print(f'{label}: not yet downloaded.')

## 3. File inventory

In [ ]:
print(f'ERA5 directory: {ERA5_DIR}\n')
files = sorted(ERA5_DIR.glob('*.nc'))
if files:
    total = 0
    print(f'{"File":<45} {"Size":>8}')
    print('-' * 55)
    for f in files:
        size_mb = f.stat().st_size / 1e6
        total  += size_mb
        status  = '✓' if size_mb > 1 else '?'
        print(f'  {status} {f.name:<43} {size_mb:>6.0f} MB')
    print('-' * 55)
    print(f'  Total: {total/1e3:.2f} GB')
else:
    print('No .nc files found — run downloads above first.')

print()
needed = [
    ('era5_omega_pressure_levels.nc', 'boundary_fixes.ipynb (Fix B stitching)'),
    ('era5_single_levels.nc',         'RQ2 environmental context'),
    ('era5_wind_200_850.nc',          'RQ2 environmental context'),
]
print('Required files:')
for fname, use in needed:
    exists = (ERA5_DIR / fname).exists()
    mark   = '✓' if exists else '✗ MISSING'
    print(f'  {mark}  {fname:<45}  ({use})')